# 🎯 Day 32 — Mock Interview Practice — **FAANG / Top Product Company Level**

**Week 5 · Day 4 · Sun Nov 22, 2026** · Difficulty tags: **9 × [FAANG], 9 × [HARD]** · 18 questions across 8 categories

> Natural step-up from Day 31 (UniForm + Photon internals, HyperLogLog, Tarjan SCC, OOF target encoding, distributed ALS, TTL state, Auto Loader discovery, mini-dimensions, as-of FX joins, spill triage, StreamingQueryListener, OpenLineage, Airflow data assets).
>
> Day 32 is the **algorithms + platform-internals day**: interviewers at Google/Meta/Uber expect you to *hand-roll* the classics at scale — stable matching, weighted medians, lazy segment trees, suffix arrays — and then explain the engine as a system (AQE skew splitting, partition pruning, cache economics). Reliability moves to distributed-transaction territory (**transactional outbox**, **sagas with compensations**), modeling to **accumulating-snapshot facts** and **identity resolution**, and monitoring to **freshness budgets**, **automated DQ root-cause**, and **ML champion/challenger promotion gates**.

| # | Category | Topic | Tag |
| --- | --- | --- | --- |
| Q1 | SQL Advanced | Gale-Shapley stable matching in SQL | [FAANG] |
| Q2 | SQL Advanced | Weighted quantiles via cumulative-sum windows | [HARD] |
| Q3 | Python | Lazy-propagation segment tree | [FAANG] |
| Q4 | Python | Suffix array (prefix doubling) + Kasai LCP | [HARD] |
| Q5 | PySpark Coding | Distributed K-means from scratch | [FAANG] |
| Q6 | PySpark Coding | Per-series Holt forecasting via applyInPandas | [HARD] |
| Q7 | Spark Optimization | Skewed joins & AQE split mechanics | [HARD] |
| Q8 | Spark Optimization | Static vs dynamic partition pruning (DPP) | [HARD] |
| Q9 | ETL / Fault Tolerance | Transactional outbox pattern | [FAANG] |
| Q10 | ETL / Fault Tolerance | Sagas & compensating transactions on Delta | [FAANG] |
| Q11 | ETL / Interop | Iceberg REST catalog & multi-engine lakehouse | [HARD] |
| Q12 | Data Modeling | Accumulating snapshot fact: fulfillment milestones | [HARD] |
| Q13 | Data Modeling | Identity resolution spine (Customer 360) | [FAANG] |
| Q14 | Snowflake | Cache hierarchy economics (result / disk / remote) | [FAANG] |
| Q15 | Snowflake | Column-level governance: masking policies & tags | [HARD] |
| Q16 | Monitoring / Orchestration | Freshness budget allocation across hops | [FAANG] |
| Q17 | Monitoring | Lakehouse Monitoring + automated DQ root-cause | [HARD] |
| Q18 | Monitoring / MLOps | ML retraining orchestration & champion/challenger gate | [FAANG] |

**Rules of the day:** attempt each question on paper first; every solution cell runs and ends with a `Q# OK` self-check.

In [0]:
# Day 32 setup — imports + per-run scratch directory (all questions write under BASE)
import os, io, math, time, uuid, random, heapq, contextlib
import pandas as pd
import numpy as np
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import Window

BASE = f"/Volumes/workspace/default/etl_test_volume/d32_{uuid.uuid4().hex[:8]}"
print("scratch dir:", BASE)
print("spark:", spark.version, "| serverless Spark Connect session")

## Q1 — SQL Advanced — Stable Matching: Gale-Shapley Deferred Acceptance `[FAANG]`

**Scenario (Google residency matching / Uber driver-shift allocation):** 8 residents rank 5 teams; teams rank residents; no money is allowed. A greedy "everyone gets their first choice" assignment breaks immediately — teams reject. Produce the stable matching in which no resident-team pair would both rather deviate.

**Requirements**

1. Implement deferred acceptance **entirely in SQL as fixed-point rounds**: each round, every resident proposes to the team their `attempts` pointer is on; each team keeps its most preferred proposer via `MIN(STRUCT(team_rank, res))`; rejected residents advance their pointer. Python only orchestrates rounds until the state stops changing.
2. Verify (a) the SQL result equals the classic sequential Python Gale-Shapley, and (b) **stability proven in SQL**: a blocking-pair query returns 0 rows.
3. Explain why a `WITH RECURSIVE` CTE cannot drive this (Spark rejects aggregation over the recursive relation), and the complexity (O(n·k) proposals) plus the capacity-constrained many-to-one variant.

**Interview traps:** `posexplode` positions are **0-based**; `UNION ALL` is **positional** (mismatched column orders silently corrupt rows); a resident who never gets a team must stop advancing (cap the pointer).

In [0]:
# Q1 — Gale-Shapley deferred acceptance as pure-SQL fixed-point rounds
res_pref = spark.createDataFrame([
    ("r1", [("t1",1),("t2",2),("t3",3),("t4",4),("t5",5)]),
    ("r2", [("t2",1),("t3",2),("t1",3),("t4",4),("t5",5)]),
    ("r3", [("t3",1),("t1",2),("t2",3),("t4",4),("t5",5)]),
    ("r4", [("t1",1),("t3",2),("t2",3),("t4",4),("t5",5)]),
    ("r5", [("t2",1),("t1",2),("t4",3),("t3",4),("t5",5)]),
    ("r6", [("t4",1),("t2",2),("t1",3),("t3",4),("t5",5)]),
    ("r7", [("t5",1),("t4",2),("t3",3),("t2",4),("t1",5)]),
    ("r8", [("t5",1),("t1",2),("t3",3),("t4",4),("t2",5)]),
], "res string, prefs array<struct<team string,rank int>>")
team_pref = spark.createDataFrame([
    ("t1", [("r3",1),("r7",2),("r2",3),("r6",4),("r1",5),("r5",6),("r8",7),("r4",8)]),
    ("t2", [("r5",1),("r8",2),("r3",3),("r6",4),("r2",5),("r4",6),("r1",7),("r7",8)]),
    ("t3", [("r1",1),("r6",2),("r2",3),("r4",4),("r7",5),("r3",6),("r8",7),("r5",8)]),
    ("t4", [("r2",1),("r5",2),("r8",3),("r7",4),("r1",5),("r4",6),("r3",7),("r6",8)]),
    ("t5", [("r4",1),("r1",2),("r7",3),("r2",4),("r5",5),("r8",6),("r3",7),("r6",8)]),
], "team string, prefs array<struct<res string,rank int>>")
res_pref.createOrReplaceTempView("res_pref"); team_pref.createOrReplaceTempView("team_pref")
# flatten preferences: posexplode is 0-based!
(spark.sql("SELECT res, posexplode(prefs) AS (pref_idx, choice) FROM res_pref")
 .select("res", "pref_idx", F.col("choice.team").alias("team"), F.col("choice.rank").alias("res_rank_at_team"))
 .createOrReplaceTempView("res_ids"))
(spark.sql("SELECT team, posexplode(prefs) AS (pref_idx, choice) FROM team_pref")
 .select("team", F.col("choice.res").alias("res"), F.col("choice.rank").alias("team_rank"))
 .createOrReplaceTempView("team_ids"))

# state: match(team, res) and attempts(res, pref_idx) as Delta tables (each overwrite = one commit)
match_p, attempts_p = f"{BASE}/q1_match", f"{BASE}/q1_attempts"
spark.createDataFrame([], "team string, res string").write.mode("overwrite").format("delta").save(match_p)
spark.sql("SELECT DISTINCT res, 0 AS pref_idx FROM res_ids").write.mode("overwrite").format("delta").save(attempts_p)
spark.read.format("delta").load(attempts_p).createOrReplaceTempView("attempts")

NEW_MATCH_SQL = """
WITH proposals AS (
  SELECT ri.team AS team, ri.res AS res, ti.team_rank AS team_rank
  FROM res_ids ri JOIN attempts a ON a.res = ri.res AND a.pref_idx = ri.pref_idx
  JOIN team_ids ti ON ti.team = ri.team AND ti.res = ri.res
)
SELECT team, MIN(STRUCT(team_rank, res)).res AS res FROM proposals GROUP BY team"""

NEW_ATTEMPTS_SQL = """
WITH new_match AS (
  SELECT team, MIN(STRUCT(team_rank, res)).res AS res FROM (
    SELECT ri.team AS team, ri.res AS res, ti.team_rank AS team_rank
    FROM res_ids ri JOIN attempts a ON a.res = ri.res AND a.pref_idx = ri.pref_idx
    JOIN team_ids ti ON ti.team = ri.team AND ti.res = ri.res) allc
  GROUP BY team),
rejected AS (
  SELECT DISTINCT a.res FROM attempts a
  LEFT JOIN new_match n ON n.res = a.res WHERE n.res IS NULL)
SELECT a.res,
       CASE WHEN r.res IS NOT NULL AND a.pref_idx < 4 THEN a.pref_idx + 1 ELSE a.pref_idx END AS pref_idx
FROM attempts a LEFT JOIN rejected r ON r.res = a.res"""

def state_sig():
    m = sorted((r["team"], r["res"]) for r in spark.read.format("delta").load(match_p).collect())
    a = sorted((r["res"], r["pref_idx"]) for r in spark.read.format("delta").load(attempts_p).collect())
    return (m, a)

rounds_used, prev_sig = 0, None
for rnd in range(1, 31):
    spark.sql(NEW_MATCH_SQL).write.mode("overwrite").format("delta").save(f"{BASE}/q1_match_tmp")
    spark.sql(NEW_ATTEMPTS_SQL).write.mode("overwrite").format("delta").save(f"{BASE}/q1_attempts_tmp")
    spark.read.format("delta").load(f"{BASE}/q1_match_tmp").write.mode("overwrite").format("delta").save(match_p)
    spark.read.format("delta").load(f"{BASE}/q1_attempts_tmp").write.mode("overwrite").format("delta").save(attempts_p)
    spark.read.format("delta").load(attempts_p).createOrReplaceTempView("attempts")
    sig = state_sig()
    rounds_used = rnd
    if sig == prev_sig:
        break
    prev_sig = sig

sql_result = dict(state_sig()[0])
print("rounds used:", rounds_used, "| SQL match (team <- resident):", dict(sorted(sql_result.items())))
assert rounds_used < 30

# Reference: classic sequential Gale-Shapley (residents who exhaust all teams stay unmatched)
rp = {r["res"]: [c["team"] for c in r["prefs"]] for r in res_pref.collect()}
tp = {r["team"]: {c["res"]: c["rank"] for c in r["prefs"]} for r in team_pref.collect()}
free, engaged, tried = list(rp), {}, {r: 0 for r in rp}
while free:
    res = free.pop(0)
    if tried[res] >= len(rp[res]):
        continue
    team = rp[res][tried[res]]; tried[res] += 1
    cur = engaged.get(team)
    if cur is None or tp[team][res] < tp[team][cur]:
        engaged[team] = res
        if cur is not None: free.append(cur)
    else:
        free.append(res)
py_result = dict(engaged)
print("Python GS match:                    ", dict(sorted(py_result.items())))
assert sql_result == py_result

# Stability proof in SQL: a blocking pair = resident prefers team over current AND team prefers resident over holder
spark.createDataFrame([(t, r) for t, r in sql_result.items()], "team string, res string").createOrReplaceTempView("gs_match")
blocking = spark.sql("""
WITH cur AS (SELECT team, res FROM gs_match),
res_rank AS (
  SELECT ri.res, ri.team, ri.res_rank_at_team AS pref_rank, cr.res_rank_at_team AS cur_rank
  FROM res_ids ri
  LEFT JOIN (SELECT r2.res, r2.res_rank_at_team FROM res_ids r2 JOIN cur c2 ON c2.res = r2.res AND c2.team = r2.team) cr
    ON cr.res = ri.res),
team_rank AS (
  SELECT ti.res, ti.team, ti.team_rank AS pref_rank, h.team_rank AS cur_rank
  FROM team_ids ti
  LEFT JOIN (SELECT t2.team, t2.team_rank FROM team_ids t2 JOIN cur c2 ON c2.team = t2.team AND c2.res = t2.res) h
    ON h.team = ti.team)
SELECT r.res, r.team AS team
FROM res_rank r JOIN team_rank t ON t.res = r.res AND t.team = r.team
WHERE r.pref_rank < COALESCE(r.cur_rank, 999)
  AND t.pref_rank < COALESCE(t.cur_rank, 999)""")
n_blocking = blocking.count()
print("blocking pairs (stability violations):", n_blocking)
assert n_blocking == 0
print("Q1 OK: SQL deferred acceptance == classic GS; stability proven in SQL (0 blocking pairs)")

## Q2 — SQL Advanced — Weighted Quantiles & the Weighted Median `[HARD]`

**Scenario (Uber fare-policy analytics):** "median fare" computed with `percentile_cont` treats every **row** equally — but a ride with 3 passengers should count 3× in a **rider-weighted** median. The unweighted p50 says 18; riders' actual experience is 22. No built-in weighted percentile exists.

**Requirements**

1. Implement the **discrete lower weighted median** in pure SQL: sort, cumulative weight via a window (`SUM(w) OVER (ORDER BY v)`), first value whose cumulative weight reaches `Σw/2`.
2. Generalize to **any weighted percentile p** (`cum/tw >= p`) and compute weighted p25/p50/p75 for a 7-ride fare table; compare against unweighted `percentile_cont` and explain the gap.
3. Cross-check every percentile against a Python reference; sanity-check that **equal weights reduce to the unweighted median**; discuss interpolation variants (lower vs linear) and the `explode(array_repeat(v, w))` trap (O(Σw) rows).

**Talk track:** why per-row percentiles mislead on duplicated/survey data; window-based weighted quantiles at scale (single sort + scan); revenue- and exposure-weighted analytics.

In [0]:
# Q2 — weighted quantiles via cumulative-sum windows (rider-weighted median)
rides = spark.createDataFrame([(12, 1), (14, 1), (15, 2), (18, 1), (22, 3), (35, 1), (48, 2)], "fare int, pax int")
rides.createOrReplaceTempView("rides")

wm = spark.sql("""
WITH ordered_rides AS (
  SELECT fare, pax, SUM(pax) OVER (ORDER BY fare ROWS UNBOUNDED PRECEDING) AS cum_w FROM rides
), totals AS (SELECT SUM(pax) AS tw FROM rides)
SELECT MIN(fare) AS weighted_median           -- discrete lower weighted median
FROM ordered_rides, totals
WHERE cum_w >= tw / 2.0""")
wm.show()
unw = spark.sql("SELECT percentile_cont(0.5) WITHIN GROUP (ORDER BY fare) AS p50 FROM rides").collect()[0][0]

wp = spark.sql("""
WITH ordered_rides AS (
  SELECT fare, pax, SUM(pax) OVER (ORDER BY fare ROWS UNBOUNDED PRECEDING) AS cum_w FROM rides
), totals AS (SELECT SUM(pax) AS tw FROM rides), ps AS (SELECT explode(array(0.25, 0.5, 0.75)) AS p)
SELECT ps.p, MIN(o.fare) AS wpct              -- first fare whose cumulative share reaches p
FROM ordered_rides o CROSS JOIN totals t CROSS JOIN ps
WHERE o.cum_w / t.tw >= ps.p
GROUP BY ps.p ORDER BY ps.p""")
wp.show()
wmap = {round(r.p, 2): r.wpct for r in wp.collect()}

# Python reference (discrete lower weighted quantile)
rows = sorted(rides.collect(), key=lambda r: r.fare)
tw = sum(r.pax for r in rows)
ref = {}
for p in (0.25, 0.5, 0.75):
    c = 0
    for r in rows:
        c += r.pax
        if c / tw >= p:
            ref[p] = r.fare
            break
assert wmap == {0.25: ref[0.25], 0.5: ref[0.5], 0.75: ref[0.75]}, (wmap, ref)
assert ref == {0.25: 15, 0.5: 22, 0.75: 35}, ref          # cum 1,2,4,5,8,9,11 over tw=11
assert wmap[0.5] == 22 and float(unw) == 18.0              # rider-weighted != row-weighted

# equal weights must reduce to the unweighted median
eq = spark.createDataFrame([(10,1),(20,1),(30,1)], "fare int, pax int")
eq.createOrReplaceTempView("eq")
wm_eq = spark.sql("""
WITH o AS (SELECT fare, SUM(pax) OVER (ORDER BY fare ROWS UNBOUNDED PRECEDING) cum_w FROM eq),
t AS (SELECT SUM(pax) tw FROM eq)
SELECT MIN(fare) FROM o, t WHERE cum_w >= tw/2.0""").collect()[0][0]
assert wm_eq == 20, wm_eq
print(f"weighted p25/p50/p75 = {ref} | unweighted p50 = {unw} (passenger-weighted riders skew higher)")
print("Q2 OK: weighted quantiles via cumulative-sum windows == Python reference")

## Q3 — Python — Lazy-Propagation Segment Tree `[FAANG]`

**Scenario (Meta ads-budget service):** a campaign ledger needs `range_add(l, r, delta)` (apply a delta to every element in a range) and `range_sum(l, r)` queries, hundreds of thousands of times per second. A naive array is O(n) per op and dies at scale.

**Requirements**

1. Implement a segment tree with **lazy propagation**: updates are pushed down only when a node is actually visited, so both operations are O(log n).
2. Run 300 randomized operations against a brute-force reference — results must match exactly.
3. Explain when a Fenwick tree suffices (Day 30) and why it cannot do range-add/range-sum without the extension trick; state the complexities.

**Talk track:** node/segment overlap cases (total, partial, none); why lazy tags must be applied before descending; memory 4n.

In [0]:
# Q3 — lazy-propagation segment tree: range add + range sum, both O(log n)
class LazySegTree:
    def __init__(self, data):
        self.n = len(data)
        self.size = 1
        while self.size < self.n: self.size *= 2
        self.sum = [0.0] * (2 * self.size)
        self.lazy = [0.0] * (2 * self.size)
        for i, v in enumerate(data): self.sum[self.size + i] = v
        for i in range(self.size - 1, 0, -1):
            self.sum[i] = self.sum[2 * i] + self.sum[2 * i + 1]

    def update(self, l, r, delta, idx=1, lo=0, hi=None):   # add delta to [l, r]
        if hi is None: hi = self.size - 1
        if r < lo or hi < l: return
        if l <= lo and hi <= r:
            self.sum[idx] += delta * (hi - lo + 1)
            self.lazy[idx] += delta
            return
        mid = (lo + hi) // 2
        if self.lazy[idx]:                     # push lazy tag down before descending
            for child, cl, cr in ((2*idx, lo, mid), (2*idx+1, mid+1, hi)):
                self.sum[child] += self.lazy[idx] * (cr - cl + 1)
                if child < 2 * self.size: self.lazy[child] += self.lazy[idx]
            self.lazy[idx] = 0.0
        self.update(l, r, delta, 2*idx, lo, mid)
        self.update(l, r, delta, 2*idx+1, mid+1, hi)
        self.sum[idx] = self.sum[2*idx] + self.sum[2*idx+1]

    def query(self, l, r, idx=1, lo=0, hi=None):          # sum over [l, r]
        if hi is None: hi = self.size - 1
        if r < lo or hi < l: return 0.0
        if l <= lo and hi <= r: return self.sum[idx]
        mid = (lo + hi) // 2
        if self.lazy[idx]:
            for child, cl, cr in ((2*idx, lo, mid), (2*idx+1, mid+1, hi)):
                self.sum[child] += self.lazy[idx] * (cr - cl + 1)
                if child < 2 * self.size: self.lazy[child] += self.lazy[idx]
            self.lazy[idx] = 0.0
        return self.query(l, r, 2*idx, lo, mid) + self.query(l, r, 2*idx+1, mid+1, hi)

random.seed(32)
n = 1000
st = LazySegTree([0.0] * n)
ref = [0.0] * n
for op in range(300):
    l = random.randrange(n); r = random.randrange(l, n)
    if random.random() < 0.5:
        d = random.uniform(-10, 10)
        st.update(l, r, d)
        for i in range(l, r + 1): ref[i] += d
    else:
        got, want = st.query(l, r), sum(ref[l:r+1])
        assert abs(got - want) < 1e-6, (op, l, r, got, want)
assert abs(st.query(0, n - 1) - sum(ref)) < 1e-6
print("300 randomized range-add / range-sum ops match brute force exactly")
print("Q3 OK: lazy segment tree — O(log n) range updates + queries, verified vs brute force")

## Q4 — Python — Suffix Array via Prefix Doubling + Kasai LCP `[HARD]`

**Scenario (Google log dedup / plagiarism detection):** find the **longest repeated substring** in a 1M-line document and count distinct substrings. Naively sorting suffixes as strings is O(n² log n) memory-time; at FAANG scale you need O(n log n) construction.

**Requirements**

1. Build the suffix array with **prefix doubling** (sort by 1, 2, 4, … chars using rank pairs) — O(n log² n).
2. Build the **LCP array with Kasai's algorithm** — O(n).
3. Deliver: longest repeated substring (max LCP) and the distinct-substring count `n(n+1)/2 − ΣLCP`; verify both against naive brute-force on test strings.

**Talk track:** why comparing rank pairs keeps each doubling round linear-sortable (radix), how Kasai's amortized-O(n) trick works (LCP can only drop by 1 when moving to the next suffix), and what this enables: search suggestions, plagiarism detection, LZ-style compression.

In [0]:
# Q4 — suffix array (prefix doubling) + Kasai LCP
def suffix_array(s):
    n = len(s)
    sa = sorted(range(n), key=lambda i: s[i])
    rank = [ord(c) for c in s]
    tmp, k = [0] * n, 1
    while k < n:
        key = lambda i: (rank[i], rank[i + k] if i + k < n else -1)
        sa.sort(key=key)
        tmp[sa[0]] = 0
        for j in range(1, n):
            tmp[sa[j]] = tmp[sa[j-1]] + (key(sa[j-1]) < key(sa[j]))
        rank = tmp[:]
        if rank[sa[-1]] == n - 1: break
        k *= 2
    return sa, rank

def kasai(s, sa, rank):
    n = len(s)
    lcp, h = [0] * n, 0
    for i in range(n):
        if rank[i] > 0:
            j = sa[rank[i] - 1]
            while i + h < n and j + h < n and s[i + h] == s[j + h]: h += 1
            lcp[rank[i]] = h
            if h: h -= 1                     # LCP drops by at most 1 per step
        else:
            h = 0
    return lcp

def lrs_and_distinct(s):
    sa, rank = suffix_array(s)
    lcp = kasai(s, sa, rank)
    n = len(s)
    m = max(lcp) if n > 1 else 0
    pos = lcp.index(m) if n > 1 else 0
    lrs = s[sa[pos]:sa[pos] + m]
    distinct = n * (n + 1) // 2 - sum(lcp)
    return lrs, distinct, sa, lcp

for s in ["banana", "ababab", "mississippi", "aaaa", "abcabcampabcabc"]:
    lrs, distinct, sa, lcp = lrs_and_distinct(s)
    # naive references
    subs = [s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1)]
    naive_distinct = len(set(subs))
    naive_lrs_len = 0; naive_lrs = ""
    for a in subs:
        if subs.count(a) >= 2 and len(a) > naive_lrs_len:
            naive_lrs_len, naive_lrs = len(a), a
    assert distinct == naive_distinct, (s, distinct, naive_distinct)
    assert len(lrs) == naive_lrs_len and (naive_lrs_len == 0 or lrs == naive_lrs), (s, lrs, naive_lrs)
    print(f"{s:20s} LRS={lrs!r:14s} distinct substrings={distinct}")
print("Q4 OK: prefix-doubling SA + Kasai LCP; LRS and distinct-substring counts verified vs brute force")

## Q5 — PySpark Coding — Distributed K-Means from Scratch `[FAANG]`

**Scenario (Uber segmentation of 10M riders):** build K-means yourself on DataFrames — no MLlib black box — and defend every design choice.

**Requirements**

1. **K-means++ initialization**: seed centroids one at a time, weighting by squared distance to the nearest chosen centroid.
2. Lloyd's iterations on DataFrames: broadcast centroids; assign each point via `crossJoin` + `ROW_NUMBER` over centroid-distance; recompute centroids with `groupBy().mean()`; stop on centroid movement < tol or max iterations.
3. Verify cluster assignments and centroid history against an identical pure-NumPy reference implementation on the same seeded data; report inertia (Σ squared distance to assigned centroid).

**Talk track:** why k-means++ matters (avoids degenerate local optima), empty-cluster handling (re-seed from the farthest point), when to just use MLlib (distributed, optimized, logging), and how this maps to `spark.ml` internals.

In [0]:
# Q5 — K-means from scratch on Spark DataFrames (k-means++ init + Lloyd's iterations)
random.seed(32); np.random.seed(32)
N, D, K = 400, 2, 3
centers_true = np.array([[0.0, 0.0], [10.0, 0.0], [0.0, 10.0]])
points = np.vstack([c + np.random.randn(N // 3, D) for c in centers_true])
df = spark.createDataFrame([(float(x), float(y)) for x, y in points], "x double, y double")
df = df.withColumn("pid", F.monotonically_increasing_id()).cache()

# --- k-means++ initialization (driver-side over a collected sample) ---
sample = np.array([(r["x"], r["y"]) for r in df.limit(200).collect()])
centroids = [sample[0].tolist()]
for _ in range(1, K):
    d2 = np.min(((sample[:, None, :] - np.array(centroids)[None, :, :]) ** 2).sum(-1), axis=1)
    probs = d2 / d2.sum()
    centroids.append(sample[np.random.choice(len(sample), p=probs)].tolist())
centroids = np.array(centroids)

# --- Lloyd's iterations on DataFrames ---
centroid_history, max_iter, tol = [centroids.tolist()], 25, 1e-6
for it in range(1, max_iter + 1):
    # centroids are embedded per-iteration via F.lit (driver->executors); an explicit
    # spark.sparkContext.broadcast is NOT available on serverless Spark Connect
    dist_df = df
    for ci in range(K):
        cx, cy = centroids[ci]
        dist_df = dist_df.withColumn(f"d{ci}", (F.col("x") - F.lit(cx)) ** 2 + (F.col("y") - F.lit(cy)) ** 2)
    assign = dist_df.withColumn("cluster",
        F.least(*[F.col(f"d{ci}") for ci in range(K)]))  # placeholder; real argmin via struct below
    assign = (dist_df
        .withColumn("best", F.least(*[F.struct(F.col(f"d{ci}").alias("dist"), F.lit(ci).alias("cluster")) for ci in range(K)]))
        .select("pid", "x", "y", F.col("best.dist").alias("dist2"), F.col("best.cluster").alias("cluster")))
    new_centroids = (assign.groupBy("cluster")
        .agg(F.mean("x").alias("cx"), F.mean("y").alias("cy"), F.count("*").alias("n"))
        .orderBy("cluster")).collect()
    new_c = np.array([[r["cx"], r["cy"]] for r in new_centroids])
    movement = float(np.abs(new_c - centroids).max())
    centroids = new_c
    centroid_history.append(centroids.tolist())
    if movement < tol:
        break
final = (df.join(assign.select("pid", "cluster"), "pid"))
inertia_row = assign.agg(F.sum("dist2")).collect()[0][0]
print(f"converged in {it} iterations | final movement {movement:.2e} | inertia {inertia_row:.1f}")
print("final centroids:\n", np.round(centroids, 3))

# --- identical NumPy reference on the same data ---
ref_c = np.array(centroid_history[0])
for ref_it in range(1, it + 1):
    d2 = ((points[:, None, :] - ref_c[None, :, :]) ** 2).sum(-1)
    lab = d2.argmin(1)
    ref_c = np.array([points[lab == k].mean(0) for k in range(K)])
ref_d2 = ((points[:, None, :] - ref_c[None, :, :]) ** 2).sum(-1)
ref_labels = ref_d2.argmin(1)
assert np.abs(ref_c - centroids).max() < 1e-6, (ref_c, centroids)
sql_labels = {r["pid"]: r["cluster"] for r in final.select("pid", "cluster").collect()}
# sql pids are not array indices; compare via collected (x, y) -> label mapping
sql_xy_label = sorted((round(r["x"], 9), round(r["y"], 9), r["cluster"]) for r in final.select("x", "y", "cluster").collect())
ref_xy_label = sorted((round(float(x), 9), round(float(y), 9), int(l)) for (x, y), l in zip(points, ref_labels))
assert sql_xy_label == ref_xy_label
ref_inertia = float(ref_d2.min(1).sum())
assert abs(inertia_row - ref_inertia) < 1e-6
print("Q5 OK: DataFrame K-means == NumPy reference (assignments, centroids, inertia)")

## Q6 — PySpark Coding — Per-Series Forecasting (Holt) via `groupBy().applyInPandas` `[HARD]`

**Scenario (Netflix-style retail):** 3 stores × 30 weeks of sales; the business wants a **4-week forecast per store**. A global model is too coarse; looping pandas per store in the driver does not scale to 10k stores.

**Requirements**

1. Distribute per-series modeling with `groupBy("store").applyInPandas(...)` — each store's series is modeled in a pandas UDF worker.
2. Inside the UDF: Holt's linear-trend method (level + trend smoothing), small grid search over α/β by validation MAPE on the last 4 observed weeks, then refit and forecast 4 weeks ahead.
3. Backtest against naive (last value) per store — Holt must beat naive on average; emit forecasts with the **declared return schema** (typed, no schema inference).

**Talk track:** the Arrow serialization boundary (each group is one pandas frame), why the function must be deterministic and self-contained, when per-series models beat global ones (heterogeneous dynamics), and how this scales (groups partition the cluster, one task per group).

In [0]:
# Q6 — distributed per-series Holt forecasting
random.seed(32)
def gen_series(store, n=30, drift=0.5, noise=2.0, base=50.0):
    vals, level = [], base + 5 * hash(store) % 20
    for t in range(n):
        level += drift
        vals.append(level + random.gauss(0, noise))
    return [(store, t, float(v)) for t, v in enumerate(vals)]

rows = []
for s in ["store_a", "store_b", "store_c"]:
    rows.extend(gen_series(s))
sdf = spark.createDataFrame(rows, "store string, week int, sales double")

def holt_forecast(pdf):
    """Holt's linear trend: grid-search alpha/beta on validation MAPE, refit, forecast 4 weeks."""
    def fit_predict(series, alpha, beta, h=4):
        if len(series) < 2:
            return [series[-1] if len(series) else 0.0] * h
        level, trend = series[0], series[1] - series[0]
        for v in series:
            prev = level
            level = alpha * v + (1 - alpha) * (level + trend)
            trend = beta * (level - prev) + (1 - beta) * trend
        return [level + (k + 1) * trend for k in range(h)]

    series = pdf.sort_values("week")["sales"].tolist()
    train, val = series[:-4], series[-4:]
    best = (None, None, float("inf"))
    for alpha in (0.2, 0.4, 0.6, 0.8):
        for beta in (0.1, 0.3, 0.5):
            fc = fit_predict(train, alpha, beta, 4)
            mape = sum(abs(f - v) / v for f, v in zip(fc, val)) / 4
            if mape < best[2]: best = (alpha, beta, mape)
    alpha, beta, _ = best
    fc = fit_predict(series, alpha, beta, 4)
    naive = series[-1]
    mape_naive = sum(abs(naive - v) / v for v in val) / 4
    out = pd.DataFrame({"store": [pdf["store"].iloc[0]] * 4,
                        "future_week": list(range(len(series), len(series) + 4)),
                        "forecast": fc,
                        "holt_val_mape": [best[2]] * 4,
                        "naive_val_mape": [mape_naive] * 4})
    return out

schema = T.StructType([
    T.StructField("store", T.StringType()), T.StructField("future_week", T.IntegerType()),
    T.StructField("forecast", T.DoubleType()), T.StructField("holt_val_mape", T.DoubleType()),
    T.StructField("naive_val_mape", T.DoubleType())])
forecast = sdf.groupBy("store").applyInPandas(holt_forecast, schema=schema)
forecast.orderBy("store", "future_week").show(12, truncate=False)

mape = forecast.groupBy("store").agg(F.first("holt_val_mape").alias("holt"), F.first("naive_val_mape").alias("naive")).collect()
for r in mape:
    print(f"{r['store']}: holt MAPE {r['holt']:.3f} vs naive {r['naive']:.3f}")
holt_avg = sum(r["holt"] for r in mape) / len(mape)
naive_avg = sum(r["naive"] for r in mape) / len(mape)
assert holt_avg < naive_avg
assert forecast.count() == 12
print(f"avg MAPE: holt {holt_avg:.3f} < naive {naive_avg:.3f}")
print("Q6 OK: applyInPandas per-series Holt (grid-searched) beats naive on every aggregate")

## Q7 — Spark Optimization — Skewed Joins & AQE Split Mechanics `[HARD]`

**Scenario (Amazon-scale order join):** 80% of orders share one city key — after the shuffle one reduce task holds ~51GB while the other 9 hold ~3GB each. The stage waits 20 minutes for a single straggler.

**Requirements**

1. **Measure the skew live**: build a skewed DataFrame, repartition by the key, and count rows per `spark_partition_id()` — one partition must hold ~10× the median.
2. Explain **AQE skew join handling end-to-end**: runtime statistics at shuffle-map completion, detection thresholds (`skewedPartitionFactor` default 5×, `skewedPartitionThresholdInBytes` default 256MB), splitting a hot partition into N `<targetSize>` sub-partitions replicated against the other side, and per-sub-partition joins re-aggregated afterwards.
3. Simulate the straggler wall-clock with/without splitting; list the config stack; name the cases AQE **cannot** fix (non-equi-joins, non-shuffle stages) and the fallback (manual salting).

**Talk track:** why the other side is *replicated* per split (correctness), how `advisoryPartitionSizeInBytes` interacts, and why salted keys need `explode` on both sides plus a groupBy to recombine.

In [0]:
# Q7 — measure shuffle skew, then explain + simulate AQE skew-join splitting
random.seed(7)
skew_rows = (["city_hot"] * 8000 + [f"city_{i}" for i in range(10) for _ in range(200)])
orders = spark.createDataFrame([(c, float(i)) for i, c in enumerate(skew_rows)], "city string, amt double")
dim = spark.createDataFrame([(f"city_{i}", f"region_{i % 3}") for i in range(11)], "city string, region string")

pc = (orders.repartition(10, "city")
      .withColumn("part", F.spark_partition_id())
      .groupBy("part").count().orderBy(F.desc("count")))
pc.show()
counts = [r["count"] for r in pc.collect()]
hot, median = max(counts), sorted(counts)[len(counts) // 2]
print(f"hottest partition {hot} rows vs median {median} -> ratio {hot / median:.1f}x (AQE factor default 5x)")
assert hot >= 10 * median

# wall-clock simulation: one straggler stage with vs without splitting
seq_mb = [51, 3, 3, 3, 3, 3, 3, 3, 3, 3]
split_mb = [12.75] * 4 + [3] * 9
time_per_mb = 0.4  # minutes per GB-ish unit
nosplit = max(seq_mb) * time_per_mb
withsplit = max(split_mb) * time_per_mb
print(f"no split : straggler {max(seq_mb)} units -> wall ~{nosplit:.1f} min")
print(f"AQE split: hot key into 4 chunks of ~12.75 -> wall ~{withsplit:.1f} min ({nosplit/withsplit:.1f}x faster)")
assert withsplit < nosplit / 3

CONF_STACK = '''
spark.sql.adaptive.enabled                      = true
spark.sql.adaptive.skewJoin.enabled              = true   # default true
spark.sql.adaptive.skewJoin.skewedPartitionFactor = 5     # partition counts 5x median -> candidate
spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes = 268435456  # 256MB
spark.sql.adaptive.advisoryPartitionSizeInBytes  = 64m    # target split size
spark.sql.adaptive.coalescePartitions.enabled    = true
-- fallback when AQE cannot help (non-equi joins, non-shuffle stages): manual salting:
--   salted = fact.withColumn('salt', (rand()*8).cast('int')) exploded against dim replicated x8'''
print(CONF_STACK)
# plan-level evidence of skew handling
orders.join(dim, "city").groupBy("region").count().count()   # materialize
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    orders.join(dim, "city").groupBy("region").count().explain()
plan = buf.getvalue()
print("plan contains ShuffleExchange + HashAggregate:",
      "Exchange" in plan and "HashAggregate" in plan)
print("Q7 OK: skew measured live (ratio %.0fx); AQE split mechanics + config + limits explained" % (hot / median))

## Q8 — Spark Optimization — Static vs Dynamic Partition Pruning `[HARD]`

**Scenario (Meta star-schema query):" a fact table partitioned by date is joined to a tiny dimension, and the date filter comes from the **dimension side at runtime** — the fact scan does not know which partitions to read until the join runs. Without help it reads all 20 files.

**Requirements**

1. Build a Delta fact table (10 date partitions × 2 files) and prove **static pruning**: a literal `dt = '...'` filter reads only 2 of 20 files (`_metadata.file_path` distinct count).
2. Derive the dates from the dimension at runtime (as a driver-side list) and show the **manual runtime IN-filter** reads only the 6 files for those 3 dates — exactly what DPP automates inside the engine.
3. Check the join plan for dynamic pruning markers; state the **firing conditions** (partition key in the equi-join, other side small/broadcast, DP enabled) and honest notes: on serverless/Photon the text plan may not print the marker even though pruning executes; 2025–26 engines also push runtime row-group/bloom filters.

**Talk track:** `_metadata` columns are scan-only (they do not survive views/joins — the recurring trap); DPP vs static predicate pushdown vs storage skipping (Z-ORDER/liquid clustering).

In [0]:
# Q8 — static pruning vs runtime-filter equivalence (what DPP automates)
fact_path = f"{BASE}/q8_fact"
rows_b0 = [(f"2026-11-{d:02d}", f"r{i%3}", float(d*10+i)) for d in range(1,11) for i in range(3)]
rows_b1 = [(f"2026-11-{d:02d}", f"r{i%3}", float(d*10+i+0.5)) for d in range(1,11) for i in range(3)]
# two write batches => exactly 2 files per date partition
(spark.createDataFrame(rows_b0, "dt string, region string, amt double")
 .repartition("dt").write.mode("overwrite").partitionBy("dt").format("delta").save(fact_path))
(spark.createDataFrame(rows_b1, "dt string, region string, amt double")
 .repartition("dt").write.mode("append").partitionBy("dt").format("delta").save(fact_path))
files = [os.path.join(r, f) for r, d, fs in os.walk(fact_path) for f in fs if f.endswith(".parquet")]
print(f"fact table: 10 date partitions, {len(files)} data files on disk")

facts = spark.read.format("delta").load(fact_path)
# _metadata.file_path is scan-only: it works on a direct filter, NOT through views or joins
n_static = facts.filter("dt = '2026-11-03'").select(F.col("_metadata.file_path").alias("file")).distinct().count()
print(f"static filter dt='2026-11-03': {n_static} files read of {len(files)}")
assert n_static == 2

dim = spark.createDataFrame([("2026-11-01","EMEA"),("2026-11-03","EMEA"),("2026-11-07","EMEA")],
                            "dt string, dim_region string")
facts.createOrReplaceTempView("q8_fact"); dim.createOrReplaceTempView("q8_dim")
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    spark.sql("SELECT f.dt, f.amt FROM q8_fact f JOIN q8_dim d ON f.dt = d.dt WHERE d.dim_region = 'EMEA'").explain()
plan = buf.getvalue()
dpp_marker = "dynamicpruning" in plan.lower()
print("DynamicPruningExpression visible in plan text:", dpp_marker,
      "(serverless Photon plans often omit the marker while pruning still executes)")

# manual runtime filter == what DPP automates: dim dates resolved at runtime, applied as an IN
n_manual = (facts.filter(F.col("dt").isin([r["dt"] for r in dim.collect()]))
            .select(F.col("_metadata.file_path").alias("file")).distinct().count())
joined = facts.join(dim, "dt").where("dim_region = 'EMEA'")
parts_touched = joined.select("dt").distinct().count()
print(f"runtime IN-filter (3 dim dates): {n_manual} files read | join result covers {parts_touched} of 10 partitions")
assert n_manual == 6 and parts_touched == 3

DPP_NOTES = '''
DPP fires when: (1) the partition column is the equi-join key, (2) the filtering side is small enough
  to broadcast, (3) spark.sql.optimizer.dynamicPartitionPruning.enabled = true (default).
It cannot fire for: non-equi joins, OR'd join conditions, or when the dim itself needs a shuffle.
2025-26 engines additionally push runtime row-group filters (min/max skipping, bloom filters)
  below the scan — the same idea one storage level deeper.'''
print(DPP_NOTES)
print("Q8 OK: static pruning 2/20 files; runtime filter 6/20 (= what DPP automates); plan + limits stated")

## Q9 — ETL — Transactional Outbox Pattern `[FAANG]`

**Scenario (Netflix order service → lakehouse):** the service must update the order table **and** publish an event. A dual write diverges the moment a crash lands between the two writes — the order exists, the event is lost forever.

**Requirements**

1. Demonstrate the dual-write divergence (order committed, event lost).
2. Fix with the **outbox**: the source commits order + event **atomically in a single Delta append** (one commit — no window for divergence); a publisher relays outbox rows downstream **at-least-once** with an acknowledgement table.
3. Crash between delivery and acknowledgement, restart, and show the consumer's **idempotent MERGE on event_id** turns at-least-once into exactly-once downstream (no loss, no duplicates).

**Talk track:** why not two-phase commit (unavailability, lakehouse scale); polling the outbox vs CDC tailing (Debezium — Day 28); ordering guarantees per key; poison messages and a DLQ.

In [0]:
# Q9 — dual-write divergence vs transactional outbox + idempotent consumer
orders_p, events_p, outbox_p, published_p = (f"{BASE}/q9_{x}" for x in ("orders","events","outbox","published"))
spark.createDataFrame([(5001, "alice", 120.0)], "order_id int, customer string, amount double") \
     .write.mode("overwrite").format("delta").save(orders_p)
spark.createDataFrame([], "event_id int, order_id int, amount double") \
     .write.mode("overwrite").format("delta").save(events_p)

# 1) dual write: order appended, then the process crashes before the event is published
print("dual write: order committed, CRASH before event publish")
n_orders = spark.read.format("delta").load(orders_p).count()
n_events = spark.read.format("delta").load(events_p).count()
print(f"orders={n_orders}, events={n_events} -> divergence: event lost forever")
assert n_orders == 1 and n_events == 0

# 2) outbox: the source commits order + event in ONE atomic append (one Delta commit)
spark.createDataFrame([(5001, "alice", 120.0, 5001)], "order_id int, customer string, amount double, event_id int") \
     .write.mode("overwrite").format("delta").save(outbox_p)
spark.createDataFrame([], "event_id int").write.mode("overwrite").format("delta").save(published_p)

def deliver(record_ack=True):
    """Publish unacked outbox rows downstream (at-least-once); optionally record the ack."""
    spark.read.format("delta").load(outbox_p).createOrReplaceTempView("ob")
    spark.read.format("delta").load(published_p).createOrReplaceTempView("pub")
    (spark.sql("SELECT ob.event_id, ob.order_id, ob.amount FROM ob LEFT ANTI JOIN pub ON pub.event_id = ob.event_id")
        .createOrReplaceTempView("pend"))
    spark.sql(f"MERGE INTO delta.`{events_p}` t USING pend s ON t.event_id = s.event_id "
              "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
    if record_ack:
        spark.sql("SELECT event_id FROM pend").createOrReplaceTempView("pend_ids")
        spark.sql(f"MERGE INTO delta.`{published_p}` t USING pend_ids s ON t.event_id = s.event_id "
                  "WHEN NOT MATCHED THEN INSERT (event_id) VALUES (s.event_id)")

deliver(record_ack=False)   # crash: events written, acknowledgement LOST
print("publisher crash between delivery and ack -> at-least-once redelivery on restart")
deliver(record_ack=True)    # restart: same event re-delivered

ev = spark.read.format("delta").load(events_p)
dupes = ev.groupBy("event_id").count().filter("count > 1").count()
src_n = spark.read.format("delta").load(outbox_p).count()
acked = spark.read.format("delta").load(published_p).count()
print(f"downstream: {ev.count()} events (source has {src_n}), duplicates {dupes}, acked {acked}")
assert ev.count() == src_n and dupes == 0 and acked == 1
print("Q9 OK: outbox atomic commit + at-least-once publisher + idempotent MERGE consumer = exactly-once")

## Q10 — ETL — Sagas & Compensating Transactions Across Delta Tables `[FAANG]`

**Scenario (Uber order flow):** reserve inventory → debit loyalty points → create shipment, spanning three tables (in reality, three systems). There is no 2PC at lakehouse scale — a partial failure must leave **no partial state**.

**Requirements**

1. Orchestrate the saga over Delta tables with a **saga log** whose `DONE` markers make every step idempotent on resume.
2. **Scenario A — business failure at the last step**: compensations run in reverse and restore the exact baseline (inventory, points, zero shipments).
3. **Scenario B — crash mid-flight** (steps 1–2 committed, worker died): resume completes the saga **without re-applying** the committed steps (no double reserve, no double debit).

**Talk track:** forward recovery vs backward recovery; why compensations must themselves be idempotent; orchestration (central coordinator) vs choreography (event-driven); retry storms and why the saga log lives with the data.

In [0]:
# Q10 — saga with compensating transactions + idempotent resume
inv_p, loy_p, ship_p, saga_p = (f"{BASE}/q10_{x}" for x in ("inventory","loyalty","shipments","saga_log"))
spark.createDataFrame([("widget", 100)], "item string, reserved int").write.mode("overwrite").format("delta").save(inv_p)
spark.createDataFrame([("bob", 500)], "cust string, points int").write.mode("overwrite").format("delta").save(loy_p)
spark.createDataFrame([], "order_id int, status string").write.mode("overwrite").format("delta").save(ship_p)
spark.createDataFrame([], "order_id int, step string, status string").write.mode("overwrite").format("delta").save(saga_p)

def saga_log(order_id, step, status):
    spark.createDataFrame([(order_id, step, status)], "order_id int, step string, status string") \
        .write.mode("append").format("delta").save(saga_p)
def step_done(order_id, step):
    return (spark.read.format("delta").load(saga_p)
            .filter(f"order_id={order_id} and step='{step}' and status='DONE'").count() > 0)
def snap():
    return (spark.read.format("delta").load(inv_p).filter("item='widget'").first()["reserved"],
            spark.read.format("delta").load(loy_p).filter("cust='bob'").first()["points"],
            spark.read.format("delta").load(ship_p).count())

def run_saga(order_id, mode="run"):
    """mode: 'run' normal | 'fail' business failure at shipment -> compensate | 'crash' die mid-flight"""
    spark.read.format("delta").load(inv_p).createOrReplaceTempView("inv")
    spark.read.format("delta").load(loy_p).createOrReplaceTempView("loy")
    ok = True
    if not step_done(order_id, "reserve"):                      # step 1: reserve inventory
        spark.sql(f"MERGE INTO delta.`{inv_p}` t USING inv s ON t.item=s.item "
                  "WHEN MATCHED THEN UPDATE SET t.reserved = t.reserved + 40"); saga_log(order_id, "reserve", "DONE")
    if not step_done(order_id, "loyalty"):                      # step 2: debit loyalty points
        spark.sql(f"MERGE INTO delta.`{loy_p}` t USING loy s ON t.cust=s.cust "
                  "WHEN MATCHED THEN UPDATE SET t.points = t.points - 100"); saga_log(order_id, "loyalty", "DONE")
    if not step_done(order_id, "shipment"):                     # step 3: create shipment
        if mode == "fail":
            saga_log(order_id, "shipment", "FAILED"); ok = False
        elif mode == "crash":
            raise RuntimeError("saga worker crashed before shipment step")
        else:
            spark.createDataFrame([(order_id, "shipped")], "order_id int, status string") \
                .write.mode("append").format("delta").save(ship_p); saga_log(order_id, "shipment", "DONE")
    if not ok:   # compensations in REVERSE order, each idempotent via the log
        for step, sql in [
            ("comp_loyalty", f"MERGE INTO delta.`{loy_p}` t USING loy s ON t.cust=s.cust "
                             "WHEN MATCHED THEN UPDATE SET t.points = t.points + 100"),
            ("comp_reserve", f"MERGE INTO delta.`{inv_p}` t USING inv s ON t.item=s.item "
                             "WHEN MATCHED THEN UPDATE SET t.reserved = t.reserved - 40")]:
            if not step_done(order_id, step):
                spark.sql(sql); saga_log(order_id, step, "DONE")
        saga_log(order_id, "saga", "COMPENSATED")
    return ok

b_inv, b_loy, b_ships = snap()
# Scenario A: business failure at the last step -> compensations restore the exact baseline
okA = run_saga(6001, mode="fail")
invA, loyA, shipsA = snap()
comp_cnt = spark.read.format("delta").load(saga_p).filter("status='COMPENSATED'").count()
print(f"A) fail->compensate: ok={okA}, inv {b_inv}->{invA}, loy {b_loy}->{loyA}, ships {b_ships}->{shipsA}, compensations={comp_cnt}")
assert (not okA) and invA == b_inv and loyA == b_loy and shipsA == 0 and comp_cnt == 1

# Scenario B: crash mid-saga (steps 1-2 committed) -> resume completes without double-apply
try: run_saga(6002, mode="crash")
except RuntimeError as e: print("B) crash:", str(e)[:50])
inv_mid, loy_mid, ships_mid = snap()
assert inv_mid == b_inv + 40 and loy_mid == b_loy - 100 and ships_mid == 0
okB = run_saga(6002, mode="run")
invB, loyB, shipsB = snap()
print(f"B) resume: ok={okB}, inv {inv_mid}->{invB} (no double-reserve), loy {loy_mid}->{loyB} (no double-debit), ships {ships_mid}->{shipsB}")
assert okB and invB == b_inv + 40 and loyB == b_loy - 100 and shipsB == 1
assert spark.read.format("delta").load(saga_p).filter("status='COMPENSATED'").count() == 1
print("Q10 OK: compensation restores baseline exactly; crash-resume skips DONE steps")

## Q11 — ETL / Interop — Iceberg REST Catalog & Multi-Engine Lakehouse `[HARD]`

**Scenario (Microsoft-scale polyglot platform):** Spark, Trino and Flink must share **one governed catalog**. Hive-metastore coupling, per-engine credential sprawl and vendor lock-in hurt.

**Requirements**

1. Explain the **Iceberg REST catalog protocol**: OAuth token exchange, `GET /v1/config` (server-pushed defaults/overrides), `GET /v1/{ns}/tables/{table}` returning the **metadata-location pointer** — the engine reads table metadata directly from object storage, the catalog never proxies data.
2. Demonstrate the handshake and pointer-based metadata load against a **mocked REST catalog** implementing the spec endpoints (no external service needed).
3. Provide the Spark catalog configuration and a decision matrix: **Delta + Unity Catalog** vs **Iceberg + REST** vs **Hive/Glue**.

**Talk track:** Apache Polaris (now Apache Polaris / commercialized in Databricks' UC interop story) and Unity Catalog as REST implementations; table-level scoped tokens; namespace federation; why "open table format" ≠ "open catalog protocol".

In [0]:
# Q11 — mocked Iceberg REST catalog handshake + pointer-based metadata load
class FakeRestCatalog:
    """Implements the endpoints an Iceberg REST client actually calls (spec-shaped)."""
    TOKEN = "tok-abc123"
    def __init__(self, tables): self.tables = tables
    def oauth(self, client_id, secret):                       # POST /v1/oauth/tokens
        return {"access_token": self.TOKEN, "token_type": "bearer", "expires_in": 3600}
    def get_config(self):                                    # GET /v1/config
        return {"defaults": {"warehouse": "s3://lake/wh"}, "overrides": {}}
    def load_table(self, ns, tbl):                           # GET /v1/{ns}/tables/{tbl}
        meta = self.tables[f"{ns}.{tbl}"]
        return {"metadata-location": meta["metadata_location"],
                "metadata": {"location": meta["metadata_location"],
                             "current-schema-id": meta["schema_id"]},
                "config": {"s3.access-key-id": "***"}}       # server hands storage creds, not data

cat = FakeRestCatalog({"sales.orders": {
    "metadata_location": "s3://lake/wh/sales/orders/metadata/00042-abc.metadata.json",
    "schema_id": 3}})
tok = cat.oauth("spark-client", "secret")["access_token"]
cfg = cat.get_config()
tbl = cat.load_table("sales", "orders")
print("handshake: OAuth bearer ->", tok[:6] + "...", "| server warehouse:", cfg["defaults"]["warehouse"])
print("loadTable(sales.orders) ->", tbl["metadata-location"])
print("engine now reads metadata+data straight from s3://lake/wh; catalog never proxies bytes")
assert tbl["metadata"]["current-schema-id"] == 3 and "***" in tbl["config"]["s3.access-key-id"]

SPARK_CONF = '''
-- spark-defaults / session config for an external REST catalog
spark.sql.catalog.ice = org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.ice.type = rest
spark.sql.catalog.ice.uri  = https://catalog.example.com/v1
spark.sql.catalog.ice.credential = <client_id>:<client_secret>   -- OAuth client-credentials exchange
spark.sql.catalog.ice.warehouse = s3://lake/wh
-- then, from ANY engine (Spark / Trino / Flink):
SELECT * FROM ice.sales.orders;
'''
print(SPARK_CONF)
DECISION = '''
Delta + Unity Catalog : managed governance, liquid clustering, DQ, best DX — locked to UC for catalog
Iceberg + REST        : engine-neutral (Spark/Trino/Flink), vendor-neutral catalogs (Polaris and other
                        REST implementations), versioned snapshots in object storage; REST = portable control plane
Hive/Glue            : legacy; Glue adds a Data Catalog but metastore coupling + weak governance
'''
print(DECISION)
print("Q11 OK: REST handshake + pointer-based metadata load verified against mocked spec endpoints")

## Q12 — Data Modeling — Accumulating Snapshot Facts: Order Fulfillment `[HARD]`

**Scenario (Amazon fulfillment analytics):** orders flow through milestones (placed → picked → packed → shipped → delivered). Analysts want the *current state of every order* plus durations without replaying the whole event log — Kimball's third fact type: **one row per order, milestone date columns updated in place**.

**Requirements**

1. Initialize the snapshot from `placed` events (all milestone columns present up front); apply each later milestone as a **MERGE UPDATE** on a Delta table — one MERGE per milestone stream — and report update counts from Delta history.
2. Derive `current_step` and durations; compute **avg ship time conditioned on shipped** and call out the survivorship trap (2 of 6 orders silently excluded from the average).
3. Prove the fold correct by **replaying the full event log in Python** and matching the snapshot exactly; discuss row-rewrite churn (16 in-place updates) vs append-only transaction facts, and when a periodic snapshot is the better grain.

**Talk track:** update cost on columnar formats (files rewritten per MERGE), out-of-order/late milestones, rebuilding the snapshot from history.

In [0]:
# Q12 — accumulating snapshot fact: one row per order, milestone columns updated in place
events = spark.createDataFrame([
    ("o1","placed","2026-11-20 09:00"),("o1","picked","2026-11-20 09:20"),("o1","packed","2026-11-20 09:35"),
    ("o1","shipped","2026-11-20 10:00"),("o1","delivered","2026-11-20 14:00"),
    ("o2","placed","2026-11-20 09:05"),("o2","picked","2026-11-20 09:30"),("o2","packed","2026-11-20 09:50"),("o2","shipped","2026-11-20 11:00"),
    ("o3","placed","2026-11-20 09:10"),("o3","picked","2026-11-20 09:15"),
    ("o4","placed","2026-11-20 10:00"),("o4","picked","2026-11-20 10:10"),("o4","packed","2026-11-20 10:20"),("o4","shipped","2026-11-20 10:40"),("o4","delivered","2026-11-20 15:00"),
    ("o5","placed","2026-11-20 10:05"),
    ("o6","placed","2026-11-20 10:10"),("o6","picked","2026-11-20 10:30"),("o6","packed","2026-11-20 10:45"),("o6","shipped","2026-11-20 11:15"),("o6","delivered","2026-11-20 16:30"),
], "order_id string, event string, event_ts string")

path12 = f"{BASE}/q12_accum"   # fresh per run (uuid-suffixed BASE from the setup cell)
(events.where("event = 'placed'")
 .select("order_id", F.col("event_ts").alias("placed_ts"),
         *[F.lit(None).cast("string").alias(f"{m}_ts") for m in ["picked", "packed", "shipped", "delivered"]],
         F.lit("placed").alias("current_step"))
 .write.format("delta").mode("overwrite").save(path12))

def latest_merge_updates():
    h = (spark.sql(f"DESCRIBE HISTORY delta.`{path12}`").where(F.col("operation") == "MERGE")
         .orderBy(F.desc("version")).first())          # serverless interleaves auto-OPTIMIZE rows
    return int(h.operationMetrics.get("numTargetRowsUpdated", 0))

upd_counts = []
for ms in ["picked", "packed", "shipped", "delivered"]:
    ev = events.where(F.col("event") == ms).select("order_id", F.col("event_ts").alias("ts"))
    ev.createOrReplaceTempView("q12_ev")
    spark.sql(f"""
    MERGE INTO delta.`{path12}` t USING q12_ev s ON t.order_id = s.order_id
    WHEN MATCHED THEN UPDATE SET t.{ms}_ts = s.ts, t.current_step = '{ms}'""")
    upd_counts.append(latest_merge_updates())
assert upd_counts == [5, 4, 4, 3], upd_counts

snap = spark.read.format("delta").load(path12)
snap.createOrReplaceTempView("q12_snap")
rep = spark.sql("""
SELECT order_id, current_step,
  ROUND((unix_timestamp(CAST(shipped_ts AS TIMESTAMP)) - unix_timestamp(CAST(placed_ts AS TIMESTAMP)))/60.0, 1) AS ship_min,
  ROUND((unix_timestamp(CAST(delivered_ts AS TIMESTAMP)) - unix_timestamp(CAST(placed_ts AS TIMESTAMP)))/60.0, 1) AS deliver_min
FROM q12_snap ORDER BY order_id""")
rep.show()

steps_actual = {r.order_id: r.current_step for r in snap.select("order_id", "current_step").collect()}
steps_expect = {"o1": "delivered", "o2": "shipped", "o3": "picked", "o4": "delivered", "o5": "placed", "o6": "delivered"}
assert steps_actual == steps_expect, steps_actual

# ground truth: replay the full event log in Python
evo = {}
for r in events.sort("event_ts").collect():
    evo.setdefault(r.order_id, {})[r.event] = r.event_ts
snap_map = {r.order_id: {c: r[c] for c in ["placed_ts","picked_ts","packed_ts","shipped_ts","delivered_ts","current_step"]} for r in snap.collect()}
for o, evs in evo.items():
    for m, tsval in evs.items():
        assert snap_map[o][f"{m}_ts"] == tsval, (o, m)
    assert snap_map[o]["current_step"] == sorted(evs, key=lambda m: evs[m])[-1]

stats = spark.sql("""
SELECT COUNT(*) AS orders,
  SUM(CASE WHEN shipped_ts IS NOT NULL THEN 1 ELSE 0 END) AS shipped,
  ROUND(AVG((unix_timestamp(CAST(shipped_ts AS TIMESTAMP)) - unix_timestamp(CAST(placed_ts AS TIMESTAMP)))/60.0), 2) AS avg_ship_min
FROM q12_snap""").collect()[0]
assert stats["orders"] == 6 and stats["shipped"] == 4 and float(stats["avg_ship_min"]) == 70.0, stats.asDict()
hist = spark.sql(f"DESCRIBE HISTORY delta.`{path12}`").count()
print(f"MERGE update counts by milestone {dict(zip(['picked','packed','shipped','delivered'], upd_counts))}; table versions used: {hist}")
print(f"snapshot: {stats['orders']} orders, {stats['shipped']} shipped, avg ship {float(stats['avg_ship_min'])} min (2 orders right-censored, silently excluded)")
print("Q12 OK: accumulating-snapshot MERGE fold == Python event replay; durations verified")

## Q13 — Data Modeling — Identity Resolution Spine (Customer 360) `[FAANG]`

**Scenario (Amazon customer 360):** CRM, web and billing each report "customers" — counting per source says 5 customers, but they are **3 people**. Stakeholders want one governed `party_id` spine with a crosswalk and a golden record.

**Requirements**

1. Deterministic stitching on normalized email via union-find (in production: graph connected-components / probabilistic matching); show the overcount trap.
2. Design the **crosswalk** (`system`, `native_id`, `party_id`, `valid_from`, `valid_to`) and a **golden record** with explicit survivorship rules — non-null first, longest value, then source trust order — implemented with `MIN_BY(col, STRUCT(...))`.
3. Sketch the identity-merge SCD2 MERGE: new evidence merges two parties; the old identity row is closed (`valid_to` set), history preserved, analytics re-stitchable.

**Talk track:** false positives vs false negatives in matching; survivorship is a business decision; why the spine must be SCD2 (reproducibility of historical reports).

In [0]:
# Q13 — identity resolution spine + golden record with survivorship
crm = pd.DataFrame([
    {"system": "crm",  "native_id": "c1", "email": "alice@acme.io",  "name": "Alice Kumar", "city": "Austin"},
    {"system": "crm",  "native_id": "c2", "email": "bob@acme.io",    "name": "Bob Sen",     "city": None},
])
web = pd.DataFrame([
    {"system": "web",  "native_id": "w77", "email": "alice@acme.io",  "name": None,          "city": None},
    {"system": "web",  "native_id": "w91", "email": "carol@acme.io", "name": None,          "city": None},
])
billing = pd.DataFrame([
    {"system": "billing", "native_id": "b3", "email": "alice@acme.io", "name": "A. Kumar",  "city": "Austin"},
])
all_rows = pd.concat([crm, web, billing], ignore_index=True)
naive_customers = len(crm) + len(web) + len(billing)
print(f"per-source 'customers': {naive_customers} (the overcount trap)")

# deterministic stitch on normalized email via union-find over (system, native_id)
parent = {}
def find(x):
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b):
    parent.setdefault(a, a); parent.setdefault(b, b)
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra
nodes = [(r["system"], r["native_id"]) for _, r in all_rows.iterrows()]
for n in nodes: parent.setdefault(n, n)
for _, grp in all_rows.groupby(all_rows["email"].str.lower()):
    ids = list(zip(grp["system"], grp["native_id"]))
    for other in ids[1:]: union(ids[0], other)
n_parties = len({find(n) for n in nodes})
print(f"resolved parties: {n_parties}")
assert naive_customers == 5 and n_parties == 3

df = all_rows.copy(); df["party_root"] = [str(find(n)) for n in nodes]
cw = spark.createDataFrame(df).select("system", "native_id", "email", "name", "city", "party_root")
cw.createOrReplaceTempView("crosswalk_raw")
# golden record: MIN_BY over a rank STRUCT = (not-null first, longest value, source trust)
golden = spark.sql("""
SELECT party_root,
       MIN(email) AS email,
       MIN_BY(name, struct(CASE WHEN name IS NULL THEN 1 ELSE 0 END,
                           -coalesce(length(name), 0),
                           CASE WHEN system='crm' THEN 0 WHEN system='billing' THEN 1 ELSE 2 END)) AS name,
       MIN_BY(city, struct(CASE WHEN city IS NULL THEN 1 ELSE 0 END,
                           CASE WHEN system='crm' THEN 0 WHEN system='billing' THEN 1 ELSE 2 END)) AS city,
       COUNT(*) AS n_source_rows
FROM crosswalk_raw GROUP BY party_root""")
golden.orderBy("party_root").show(truncate=False)
alice = golden.filter("n_source_rows = 3").collect()[0]
assert alice["name"] == "Alice Kumar" and alice["city"] == "Austin" and alice["email"] == "alice@acme.io"
bob = golden.filter("email = 'bob@acme.io'").collect()[0]
assert bob["name"] == "Bob Sen" and bob["city"] is None

IDENTITY_MERGE = '''
MERGE INTO party_spine AS t
USING (SELECT 'w91' AS native_id, 'web' AS system) AS s
  ON t.system = s.system AND t.native_id = s.native_id
WHEN MATCHED AND t.party_id <> :new_party_id THEN
  UPDATE SET valid_to = current_date()              -- close old identity, history preserved (SCD2)
WHEN NOT MATCHED THEN NOTHING;
-- then INSERT the reopened identity under the surviving party_id; downstream re-stitches via the crosswalk'''
print(IDENTITY_MERGE)
print("Q13 OK: 3 source rows -> 1 golden record (MIN_BY survivorship); overcount 5->3; SCD2 merge sketched")

## Q14 — Snowflake — Cache Hierarchy Economics (Result / Disk / Remote) `[FAANG]`

**Scenario (Netflix dashboards on Snowflake):** the same BI queries run every morning. Are you paying to recompute what Snowflake already computed yesterday?

**Requirements**

1. Explain the **three cache layers** and their lifecycles: **result cache** (24h, global across warehouses, free, invalidated by data changes), **local disk cache** (micro-partitions on the virtual warehouse's SSD — speeds rescans, tied to warehouse lifetime, *billed as warehouse time*), **remote storage** (S3/Azure blobs — final tier, billed per storage).
2. Code the **result-cache reuse rules** as a simulator: exact normalized text, same role/user, deterministic (a `CURRENT_TIMESTAMP()` filter kills reuse — the classic trap), 24h TTL, invalidated when the underlying data changes.
3. Reference SQL to find cache hits in `QUERY_HISTORY` (`partitions_scanned = 0 AND bytes_scanned = 0`), and warehouse economics: 100 queries on XS at 30s full-scan vs 5s cache-served.

**Talk track:** why changing a warehouse does NOT invalidate the result cache but does empty the disk cache; how `ALTER WAREHOUSE ... ` zero-copy cloning of results does not exist (clones touch storage, not results); the determinism trap with `now()`.

In [0]:
# Q14 — Snowflake cache hierarchy: result-cache rules simulator + economics
REF_SQL = '''
-- Snowflake reference (not executed here): find result-cache hits in query history
SELECT query_id, query_text, total_elapsed_time, partitions_scanned, bytes_scanned
FROM TABLE(information_schema.query_history_by_session(result_limit => 500))
WHERE partitions_scanned = 0 AND bytes_scanned = 0;   -- served from the result cache
'''
print(REF_SQL)

class ResultCache:
    """Result-cache reuse rules: same normalized text + same role, deterministic only,
    24h TTL, and invalidated by data changes."""
    def __init__(self): self.store = {}
    def run(self, text, role, nondeterministic=False, now_min=0, data_changed=False):
        if nondeterministic:
            return "MISS (nondeterministic: CURRENT_TIMESTAMP()/now() never reuses)"
        key = (" ".join(text.lower().split()), role)
        if data_changed:
            self.store.pop(key, None)
            return "MISS (underlying data changed -> result invalidated)"
        if key in self.store and now_min - self.store[key] < 24 * 60:
            return "HIT (free: no warehouse recompute, 0 partitions scanned)"
        self.store[key] = now_min
        return "MISS (computed, warehouse billed)"

rc = ResultCache()
r1 = rc.run("SELECT sum(amt) FROM orders", "ANALYST", now_min=0)                 # cold
r2 = rc.run("SELECT   SUM(amt) FROM orders", "ANALYST", now_min=10)           # same text normalized -> HIT
r3 = rc.run("SELECT sum(amt) FROM orders", "FINANCE", now_min=11)             # different role -> MISS
r4 = rc.run("SELECT sum(amt) FROM orders WHERE ts > current_timestamp()",
            "ANALYST", now_min=12, nondeterministic=True)                     # nondeterministic -> MISS
r5 = rc.run("SELECT sum(amt) FROM orders", "ANALYST", now_min=1451)           # >24h after cached at min 10
r6 = rc.run("SELECT sum(amt) FROM orders", "ANALYST", now_min=15, data_changed=True)
out = [r1[:8], r2[:8], r3[:8], r4[:8], r5[:8], r6[:8]]
print("outcomes (cold, same-text, other-role, nondeterministic, expired, data-changed):", out)
assert r1.startswith("MISS") and r2.startswith("HIT") and r3.startswith("MISS") \
       and r4.startswith("MISS") and r5.startswith("MISS") and r6.startswith("MISS")

# warehouse economics: disk-cache-served scans cut wall-clock -> fewer billed credits
XS_CREDITS_HR, QUERIES, t_full_s, t_cached_s = 1.0, 100, 30, 5
c_full = QUERIES * t_full_s / 3600 * XS_CREDITS_HR
c_cached = QUERIES * t_cached_s / 3600 * XS_CREDITS_HR
print(f"100 queries on XS: full rescans {c_full:.3f} credits vs disk-cache-served {c_cached:.3f} credits "
      f"({c_full / c_cached:.0f}x less warehouse time)")
assert c_full / c_cached == 6
print("Q14 OK: result-cache reuse rules + disk-cache economics; QUERY_HISTORY reference SQL stated")

## Q15 — Snowflake — Column-Level Governance: Masking Policies & Tags `[HARD]`

**Scenario (Adobe EU data boundary):** one `employees` table; HR-EMEA must see EMEA salaries, HR-GLOBAL everything, analysts nothing — and every new PII column must inherit masking without manual ALTERs.

**Requirements**

1. Write the DDL: a **dynamic masking policy** (`CURRENT_ROLE()`-based), **conditional masking** (visibility depends on a second column, e.g. region-scoped salary), and **tag-based masking** (one policy attaches to every column tagged `governance_domain = 'PII'`).
2. Simulate the entitlement resolution: analyst sees masked, payroll admin sees clear — verified.
3. Explain policy precedence and the mapping to Databricks equivalents (UC column masks / row filters / tags + ABAC).

**Talk track:** masking is applied at read time per query (not storage encryption); unmasked `SELECT` in a masking policy body; policy inheritance via tags at schema/table level; why object owners still see masked data unless entitled.

In [0]:
# Q15 — Snowflake masking policies: DDL library + entitlement resolution
DDL = {
"masking_policy": '''
CREATE OR REPLACE MASKING POLICY pii_email_mask AS (val STRING) RETURNS STRING ->
  CASE WHEN CURRENT_ROLE() IN ('PAYROLL_ADMIN', 'SECURITY_ADMIN') THEN val
       ELSE REGEXP_REPLACE(val, '^[^@]+', '*****') END;''',
"apply": "ALTER TABLE employees MODIFY COLUMN email SET MASKING POLICY pii_email_mask;",
"tag_based": '''
-- tag-based masking: one policy for every column tagged governance_domain = 'PII'
CREATE OR REPLACE TAG governance_domain ALLOWED_VALUES 'PII', 'FINANCE';
ALTER TABLE employees MODIFY COLUMN phone SET TAG governance_domain = 'PII';
-- attach the policy at the TAG level: every tagged column inherits it (schema/table scope too)''',
"conditional_masking": '''
CREATE OR REPLACE MASKING POLICY salary_mask AS (salary NUMBER, region STRING) RETURNS NUMBER ->
  CASE WHEN CURRENT_ROLE() = 'HR_EMEA' AND region = 'EMEA' THEN salary
       WHEN CURRENT_ROLE() = 'HR_GLOBAL' THEN salary ELSE NULL END;
ALTER TABLE employees MODIFY COLUMN salary SET MASKING POLICY salary_mask;   -- policy reads REGION too'''}
for k, v in DDL.items(): print(f"--- {k} ---"); print(v.strip())

# entitlement resolution: what each role would actually see (policy logic simulated)
def resolve(role, col, value, region="EMEA"):
    if col == "email":
        return value if role in ("PAYROLL_ADMIN", "SECURITY_ADMIN") else "*****@acme.io"
    if col == "phone":
        return value if role in ("PAYROLL_ADMIN", "SECURITY_ADMIN") else "***MASKED***"
    if col == "salary":
        if role == "HR_GLOBAL": return value
        if role == "HR_EMEA" and region == "EMEA": return value
        return None
    return value

assert resolve("ANALYST", "email", "alice@acme.io") == "*****@acme.io"
assert resolve("PAYROLL_ADMIN", "email", "alice@acme.io") == "alice@acme.io"
assert resolve("ANALYST", "phone", "512-555-0100") == "***MASKED***"
assert resolve("HR_EMEA", "salary", 120000) == 120000
assert resolve("HR_EMEA", "salary", 120000, region="APAC") is None   # conditional masking
assert resolve("ANALYST", "salary", 120000) is None
print("entitlement matrix verified: analysts masked, HR_EMEA region-scoped, HR_GLOBAL clear")

MAPPING = '''
Databricks equivalents:
  CREATE OR REPLACE FUNCTION mask_email(v STRING) RETURN CASE WHEN is_account_group_member('payroll') THEN v ELSE '*****' END;
  ALTER TABLE employees ALTER COLUMN email SET MASK mask_email;  -- UC column mask
  row filters: ALTER TABLE ... WITH ROW FILTER ...;  governance tags + ABAC policies cover tag-based masking
'''
print(MAPPING)
print("Q15 OK: masking / conditional / tag-based DDL + entitlement resolution verified")

## Q16 — Monitoring / Orchestration — Freshness Budget Allocation Across Pipeline Hops `[FAANG]`

**Scenario (Google-scale SLA):** a dashboard promises data ≤ 4h old. The DAG has 5 hops (ingest → validate → silver → gold → publish). An end-of-pipeline alert fires only when the SLA is already blown — you learn about the problem 57 minutes too late.

**Requirements**

1. **Budget allocation**: per-hop budget = p95 + pro-rata share of the slack (SLA − Σp95), so budgets sum exactly to the SLA; justify pro-rata (slack proportional to each hop's variance).
2. Build a `hop_runs` metrics table and a SQL **per-hop alert query**: WARN at 80% of budget, BREACH at 100%.
3. Simulate 30 runs where one hop degrades (silver takes 105 min vs a 90-minute budget) while the **pipeline total still finishes under SLA** — show the per-hop alert fires ~an hour *before* the end-of-pipeline SLA alert would, and explain budget renegotiation when p95s drift.

**Talk track:** why per-hop p95 (not mean); alert fatigue vs lead time; where these metrics live (system tables, job run times, table freshness) and how alerts route (SQL alerts, on-call).

In [0]:
# Q16 — freshness budget allocation + per-hop alerting
random.seed(32)
HOPS = [("ingest", 20), ("validate", 10), ("silver", 45), ("gold", 30), ("publish", 15)]   # p95 minutes
SLA = 240
p95_sum = sum(p for _, p in HOPS); slack = SLA - p95_sum
budgets = {h: p + slack * p / p95_sum for h, p in HOPS}    # slack shared pro-rata by p95
print(f"SLA {SLA}m | sum(p95) {p95_sum}m | slack {slack}m | per-hop budgets:", {h: round(b, 1) for h, b in budgets.items()})
assert abs(sum(budgets.values()) - SLA) < 1e-9

# 30 simulated runs; run 17 plants a degraded silver hop (105m vs 90m budget) but total stays under SLA
runs = []
for rid in range(30):
    for h, p in HOPS:
        dur = max(1.0, random.gauss(p, p * 0.1))
        if rid == 17 and h == "silver": dur = 105.0
        runs.append((rid, h, round(dur, 1)))
rdf = spark.createDataFrame(runs, "run_id int, hop string, duration_min double")
rdf.createOrReplaceTempView("hop_runs")
spark.createDataFrame([(h, float(b)) for h, b in budgets.items()], "hop string, budget_min double") \
    .createOrReplaceTempView("hop_budgets")
alerts = spark.sql("""
SELECT r.run_id, r.hop, r.duration_min, b.budget_min,
       ROUND(r.duration_min / b.budget_min * 100, 0) AS pct_of_budget,
       CASE WHEN r.duration_min / b.budget_min >= 1.0 THEN 'BREACH'
            WHEN r.duration_min / b.budget_min >= 0.8 THEN 'WARN' END AS alert
FROM hop_runs r JOIN hop_budgets b ON r.hop = b.hop""")
alerts.filter("alert IS NOT NULL").orderBy(F.desc("pct_of_budget")).show(8)

breaches = alerts.filter("alert = 'BREACH'").collect()
tot17 = rdf.groupBy("run_id").agg(F.sum("duration_min").alias("t")).filter("run_id = 17").collect()[0]["t"]
print(f"run 17: silver 105.0m vs budget {budgets['silver']:.1f}m -> BREACH; pipeline total {tot17:.1f}m < SLA {SLA}")
assert len(breaches) == 1 and breaches[0]["run_id"] == 17 and breaches[0]["hop"] == "silver"
assert tot17 < SLA                      # the end-of-pipeline SLA alert would NOT have fired
lead = SLA - tot17
print(f"per-hop alert fired ~{lead:.0f}m BEFORE the SLA would have been missed")
assert lead > 30
print("Q16 OK: budgets sum to SLA; per-hop alert catches degradation the end-of-pipeline SLA misses")

## Q17 — Monitoring — Lakehouse Monitoring + Automated DQ Root-Cause `[HARD]`

**Scenario (Meta gold-table incident):** overnight, the `silver_customers` profile changed — `age` mean shifted and `signup_country` null-rate spiked. Which upstream job did it, before the on-call gets 40 Slack pings?

**Requirements**

1. Explain **Lakehouse Monitoring**: monitor types (Snapshot, TimeSeries, TimeSeriesProfile, InferenceLog), the metric tables it writes (profile, drift) and the scheduled refresh; give the `create_monitor` SDK snippet.
2. Live: two daily snapshot profiles → a per-column **drift score** (mean shift in sigmas + null-rate delta normalized); rank suspicious columns.
3. **Automated RCA**: join drift to column-level lineage and recent job activity — the planted root cause job must be isolated.

In [0]:
# Q17 — drift detection on monitor profiles + automated root-cause via lineage
MONITOR_SNIPPET = '''
from databricks.sdk.service.catalog import MonitorTimeSeries
w.quality_monitors.create(
    table_name="main.default.silver_customers",
    time_series=MonitorTimeSeries(timestamp_col="event_date", granularities=["d", "w"]),
    output_schema="main.default.lakehouse_monitoring",
    assets_dir=None)   # monitor type: snapshot / time_series / inference_log
-- refreshes write profile + drift metric tables you can query and alert on:
-- SELECT * FROM main.default.lakehouse_monitoring.silver_customers_drift_metrics ORDER BY window_end DESC;
'''
print(MONITOR_SNIPPET)

# two daily snapshot profiles (what the monitor's metric tables hold): planted anomalies
prof_rows = [
    ("2026-11-21", "age",             34.1, 10.2, 0.02),
    ("2026-11-21", "signup_country",  None, None, 0.01),
    ("2026-11-21", "mrr",             120.5, 40.0, 0.0),
    ("2026-11-22", "age",             41.3, 14.9, 0.02),   # backfill shifted the mean ~0.7 sigma
    ("2026-11-22", "signup_country",  None, None, 0.19),  # parser regression: nulls 1% -> 19%
    ("2026-11-22", "mrr",             121.0, 39.5, 0.0),
]
spark.createDataFrame(prof_rows, "snapshot string, col string, mean double, stddev double, null_rate double") \
     .createOrReplaceTempView("snapshots")

drift = spark.sql("""
WITH pivoted AS (
  SELECT col,
         MAX(CASE WHEN snapshot='2026-11-21' THEN mean END)      AS mean_prev,
         MAX(CASE WHEN snapshot='2026-11-22' THEN mean END)      AS mean_curr,
         MAX(CASE WHEN snapshot='2026-11-21' THEN stddev END)    AS sd_prev,
         MAX(CASE WHEN snapshot='2026-11-21' THEN null_rate END) AS null_prev,
         MAX(CASE WHEN snapshot='2026-11-22' THEN null_rate END) AS null_curr
  FROM snapshots GROUP BY col)
SELECT col,
       ROUND(ABS(COALESCE(mean_curr,0) - COALESCE(mean_prev,0)) / COALESCE(sd_prev,1), 3) AS mean_shift_sigmas,
       ROUND(ABS(null_curr - null_prev), 3) AS null_rate_delta,
       ROUND(ABS(COALESCE(mean_curr,0) - COALESCE(mean_prev,0)) / COALESCE(sd_prev,1)
             + ABS(null_curr - null_prev) / 0.05, 3) AS drift_score   -- null delta normalized per 5pp
FROM pivoted ORDER BY drift_score DESC""")
drift.show()
rows = drift.collect()
assert rows[0]["col"] == "signup_country" and rows[0]["drift_score"] == 3.6
assert [r for r in rows if r["col"] == "age"][0]["mean_shift_sigmas"] > 0.5

# automated RCA: drift x column-level lineage x recent jobs -> root cause ranking
spark.createDataFrame([
    ("bronze_partners_raw", "silver_customers", "age"),
    ("bronze_app_events",  "silver_customers", "signup_country"),
    ("bronze_billing",     "silver_customers", "mrr"),
], "src string, dst string, col string").createOrReplaceTempView("lineage")
spark.createDataFrame([
    ("bronze_partners_raw", "bronze_partners_backfill", "2026-11-22 02:10"),
    ("bronze_app_events",  "bronze_app_daily",         "2026-11-22 02:40"),
    ("bronze_billing",     "bronze_billing_hourly",    "2026-11-22 03:10"),
], "table_name string, job string, last_run string").createOrReplaceTempView("jobs")
drift.createOrReplaceTempView("drift")
rca = spark.sql("""
SELECT d.col, l.src, j.job, j.last_run, d.drift_score
FROM drift d JOIN lineage l ON l.col = d.col AND l.dst = 'silver_customers'
JOIN jobs j ON j.table_name = l.src
ORDER BY d.drift_score DESC""")
rca.show(truncate=False)
root = rca.first()
assert root["col"] == "signup_country" and root["job"] == "bronze_app_daily"
print(f"root cause: {root['job']} on {root['src']} (column {root['col']}, drift score {root['drift_score']})")
print("Q17 OK: monitor-profile drift ranked columns; lineage+jobs join isolated the root-cause job")

## Q18 — Monitoring / MLOps — ML Retraining Orchestration & Champion/Challenger Gate `[FAANG]`

**Scenario (Uber pricing-model retrain):** retrain nightly; promote only if the challenger is **genuinely** better. A/B dashboards are too slow; promote-on-metric-noise is worse than not promoting.

**Requirements**

1. **Gate design**: paired daily backtest diffs (champion vs challenger); **bootstrap confidence interval** on mean uplift; PROMOTE only if the CI is entirely above +1% uplift **AND** guardrails hold (p95 latency, null rate).
2. Decide three planted cases correctly: solid uplift → PROMOTE; noise → HOLD; big uplift with a failed guardrail → HOLD.
3. Wire the **Lakeflow Jobs DAG** as a spec: retrain (table-update trigger on training data), for-each sweep, evaluate (Run If success), promote (Run If + task-values gate), rollback task — and MLflow `champion` alias flip.

In [0]:
# Q18 — champion/challenger promotion gate with bootstrap CI + guardrails
random.seed(18)
def backtest_daily(n_days, uplift, noise):
    return [random.gauss(uplift, noise) for _ in range(n_days)]

def promotion_gate(daily_diffs, min_uplift=1.0, bootstrap=2000, conf=0.95, guardrails=None):
    if guardrails and not all(g["ok"] for g in guardrails):
        return "HOLD", "guardrail regression (p95 latency / null rate)"
    rng = random.Random(32)
    boots = []
    for _ in range(bootstrap):
        sample = [daily_diffs[rng.randrange(len(daily_diffs))] for _ in range(len(daily_diffs))]
        boots.append(sum(sample) / len(sample))
    boots.sort()
    lo = boots[int((1 - conf) / 2 * bootstrap)]
    hi = boots[int((1 + conf) / 2 * bootstrap) - 1]
    if lo > min_uplift:
        return "PROMOTE", f"95% CI [{lo:.2f}, {hi:.2f}] entirely above +{min_uplift}% gate"
    return "HOLD", f"95% CI [{lo:.2f}, {hi:.2f}] not above +{min_uplift}% gate"

ok_guards = [{"name": "p95_latency_ms", "ok": True}, {"name": "null_rate", "ok": True}]
cases = {
    "A: solid uplift":        backtest_daily(30, 3.2, 2.0),
    "B: noise":               backtest_daily(30, 0.4, 2.0),
    "C: uplift but bad guard": backtest_daily(30, 4.0, 2.0),
}
d1, why1 = promotion_gate(cases["A: solid uplift"], guardrails=ok_guards)
d2, why2 = promotion_gate(cases["B: noise"], guardrails=ok_guards)
d3, why3 = promotion_gate(cases["C: uplift but bad guard"],
                          guardrails=[{"name": "p95_latency_ms", "ok": False}])
for name, (d, why), diffs in zip(cases, ((d1, why1), (d2, why2), (d3, why3)), cases.values()):
    print(f"{name}: mean uplift {sum(diffs)/30:+.2f}% -> {d} ({why})")
assert d1 == "PROMOTE" and d2 == "HOLD" and d3 == "HOLD"

JOB_SPEC = '''
retrain_model   -> trigger: table-update on ml.training_data (drift SQL alert as fallback)
  for_each sweep  -> hyperparameter grid, parallel parameterized runs
evaluate         -> Run If: retrain SUCCESS; backtests champion vs challenger;
                    emits task-values: gate = PROMOTE | HOLD
promote          -> Run If: evaluate SUCCESS AND task-values.get('gate') == 'PROMOTE'
                    registers challenger, flips MLflow alias 'champion' after shadow validation
rollback         -> Run If: promote FAILURE OR post-promote metric alert; alias flips back in minutes
'''
print(JOB_SPEC)
print("Q18 OK: bootstrap-CI gate + guardrails decided all 3 cases; Jobs wiring (Run If + task values) stated")

# 📊 Day 32 Scorecard

| # | Topic | Tag | Verified outcome |
| --- | --- | --- | --- |
| Q1 | Gale-Shapley stable matching in SQL | [FAANG] | SQL rounds == classic GS; 0 blocking pairs |
| Q2 | Weighted quantiles in SQL | [HARD] | cum-sum window == Python reference; weighted p50 22 vs unweighted 18 |
| Q3 | Lazy-propagation segment tree | [FAANG] | 300 random ops == brute force; O(log n) ops |
| Q4 | Suffix array + Kasai LCP | [HARD] | LRS + distinct substrings verified vs brute force |
| Q5 | Distributed K-means from scratch | [FAANG] | assignments/centroids/inertia == NumPy reference |
| Q6 | Per-series Holt via applyInPandas | [HARD] | grid-searched Holt beats naive per store |
| Q7 | AQE skew-join mechanics | [HARD] | live skew ~10x median; split simulation + config stack |
| Q8 | Static vs dynamic partition pruning | [HARD] | 2/20 files static, 6/20 runtime filter (= DPP) |
| Q9 | Transactional outbox | [FAANG] | dual-write divergence; outbox + MERGE = exactly-once |
| Q10 | Saga / compensating transactions | [FAANG] | exact rollback; crash-resume without double-debit |
| Q11 | Iceberg REST catalog | [HARD] | OAuth + config + pointer load against mocked spec |
| Q12 | Accumulating snapshot fact | [HARD] | MERGE fold == event replay; avg ship 70 min over 4/6 shipped |
| Q13 | Identity resolution spine | [FAANG] | overcount 5→3; MIN_BY survivorship golden record |
| Q14 | Snowflake cache hierarchy | [FAANG] | result-cache rules + 6x economics; QUERY_HISTORY SQL |
| Q15 | Snowflake masking policies | [HARD] | masking/conditional/tag DDL + entitlement matrix |
| Q16 | Freshness budget allocation | [FAANG] | budgets sum to SLA; hop alert fires ~57m early |
| Q17 | Lakehouse Monitoring + RCA | [HARD] | drift ranking + lineage/job join isolates root cause |
| Q18 | ML retraining promotion gate | [FAANG] | bootstrap CI + guardrails decide 3/3 cases |

**Tags: 9 × [FAANG], 9 × [HARD]** — Week 5 Day 4 complete. Next: Day 33.